In [10]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os

In [3]:
INPUT_DIR = "../../../raw/"
OUTPUT_DIR = "../../../clean"
COUNTRY_RENAME_MAPPING = {
        "NRU": "NAU", "ROU": "ROM", "SCG": "SER", "SGP": "SIN",
        "TWN": "TAW", "ARE": "UAE", "COD": "ZAR", "SRB": "SER",
        "MNE": "SER"
}
FILE_NAME_SUFFIX = "_1970_2025"
FILE_NAME_PREFIXES = ["IEA_EDGAR_CO2", "EDGAR_N2O", "EDGAR_CH4"]

In [46]:
def load_sheet_from_zip(archive: ZipFile):
        sheet_name = archive.filename.split("/")[-1].split(".")[0]
        sheet_name += ".xlsx"

        with archive.open(sheet_name) as sheet_io:
                return pd.read_excel(sheet_io, sheet_name=3, header=9)

def load_sheet_from_zip_path(prefix: str):
        zip_path = os.path.join(
                INPUT_DIR, prefix + FILE_NAME_SUFFIX + ".zip"
        )

        with ZipFile(zip_path) as archive:
                return load_sheet_from_zip(archive)

def cleaning_table(table: pd.DataFrame):
        value_vars = [col for col in table.columns if "Y_" in col]
        long_table = table.melt(
                ["Country_code_A3"], value_vars, "year_str", 
                table["Substance"].unique()[0]
        )
        year_ints = long_table["year_str"].str[2:].astype(int)
        long_table["year"] = year_ints
        long_table = (
                long_table
                .drop(columns="year_str")
                .replace(COUNTRY_RENAME_MAPPING)
                .rename(columns={"Country_code_A3": "wbcode"})
                .sort_values(["wbcode", "year"])
                .reset_index(drop=True)
        )
        return long_table

def clean_all_edgar_tables():
        for prefix in FILE_NAME_PREFIXES:
                table = load_sheet_from_zip_path(prefix)
                clean_table = cleaning_table(table)
                yield prefix, clean_table

def main():
        for prefix, table in clean_all_edgar_tables():
                pollutant_name = prefix[-3:]
                table_path = os.path.join(
                        OUTPUT_DIR, pollutant_name.lower() + ".csv"
                )
                table.to_csv(table_path, index=False)

In [47]:
if __name__ == "__main__":
        main()